In [11]:
"""
TOP1模型 (Gp_GP_Matern_0.5_K6_Comb17) SHAP完整分析程序 v9
v9修改：增强版Dependence图 Positive/Negative 分区改为精确版
  - 根据 Lowess 拟合曲线与 SHAP=0 的实际交叉点精确划分正负区域
  - 在每个交叉点处标注特征值数值（红色虚线 + 红色圆点 + 数值标注）
  - 正贡献区（浅红背景）/ 负贡献区（浅蓝背景）按实际曲线走向精确填充
  对应图2/3/4的风格

其他所有功能与v8完全一致。
输出目录: D:\\博二上\\模型分析可视化\\shap7
"""

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
plt.ioff()

import seaborn as sns
import shap
import pickle
import os
import warnings
from scipy.stats import pearsonr

warnings.filterwarnings('ignore')

matplotlib.rcParams['font.family']              = 'Arial'
matplotlib.rcParams['pdf.fonttype']             = 42
matplotlib.rcParams['ps.fonttype']              = 42
matplotlib.rcParams['axes.unicode_minus']       = False
matplotlib.rcParams['figure.dpi']              = 300
matplotlib.rcParams['savefig.dpi']             = 300
matplotlib.rcParams['font.size']               = 10
matplotlib.rcParams['figure.max_open_warning'] = 0
sns.set_style("whitegrid")

# ====================== 配置参数 ======================
CONFIG = {
    'data_file': (
        r"D:\博一下\第一章主动学习"
        r"\Nb-Si数据库10.18-26个-4种参数-成分-性能-Si小于10的全部去掉.xlsx"
    ),
    'pkl_file': (
        r"D:\博一下\第一章主动学习\高斯过程组_结果_v3.5-Extended"
        r"\top30_models\model_objects"
        r"\rank01_Gp_GP_Matern_0.5_K6_Comb17_model.pkl"
    ),
    'output_dir': r"D:\博二上\模型分析可视化\shap7",

    'selected_features': [
        'deltaP_热导率W/(mk)', 'x', 'Ec', 'Ω',
        'deltaP_E13 Electron affinity', 'ΔSmix',
    ],
    'target_col':   'KQ',
    'random_state': 2023,

    'shap_background_samples':  50,
    'shap_explain_samples':     None,
    'dpi':                      300,
    'figure_size':              (12, 8),

    'top_interaction_features': 3,
    'force_plot_samples':       5,
    'waterfall_samples':        3,

    'heatmap_sort_by':  'prediction',
    'heatmap_cmap':     'RdBu_r',

    'dependence_lowess_frac': 0.3,
    'dependence_poly_degree': 3,
    'dependence_ci_alpha':    0.95,

    # 精确分区颜色配置
    'pos_color': '#FFCCCC',   # 正贡献区背景色（浅红）
    'neg_color': '#CCE5FF',   # 负贡献区背景色（浅蓝）
    'pos_alpha': 0.40,
    'neg_alpha': 0.40,
    'cross_color': '#d62728', # 交叉点标注颜色（红色）
}

# ====================== 工具函数 ======================
def cleanup_matplotlib():
    plt.close('all')
    import gc; gc.collect()

def safe_filename(name):
    reps = {
        ' ':'_','/':'_','\\':'_',':':'_','*':'_','?':'_',
        '"':'_','<':'_','>':'_','|':'_','(':'',')':'',
        '[':'',']':'','{':'','}':'','×':'x','÷':'div',
        '+':'plus','=':'eq','%':'percent','#':'num','&':'and',
        '@':'at','$':'dollar','!':'excl','^':'caret','~':'tilde',
        '`':'grave',"'":'',',':'_',';':'_','.':'_','-':'_',
        'Δ':'Delta','δ':'delta','Ω':'Omega','Λ':'Lambda',
        'α':'alpha','β':'beta','γ':'gamma',
    }
    s = name
    for o, n in reps.items():
        s = s.replace(o, n)
    while '__' in s:
        s = s.replace('__', '_')
    return s.strip('_')[:50]

def create_output_dirs(base_dir):
    for s in ['shap_summary','shap_heatmap','shap_dependence',
              'shap_dependence_enhanced','shap_interaction',
              'shap_force','shap_waterfall']:
        os.makedirs(os.path.join(base_dir, s), exist_ok=True)
    print(f"✓ 输出目录已创建: {base_dir}")

def write_excel_multisheets(filepath, sheets_dict, info_dict=None):
    try:
        with pd.ExcelWriter(filepath, engine='openpyxl') as writer:
            for sname, df in sheets_dict.items():
                if df is not None and len(df) > 0:
                    df.to_excel(writer, sheet_name=str(sname)[:31], index=False)
            if info_dict:
                pd.DataFrame(
                    list(info_dict.items()), columns=['Parameter', 'Value']
                ).to_excel(writer, sheet_name='_Info', index=False)
        return True
    except Exception as e:
        print(f"      ❌ Excel保存失败 ({os.path.basename(filepath)}): {e}")
        return False

# ====================== Lowess + 多项式 ======================
def lowess_smooth(x, y, frac=0.3):
    idx = np.argsort(x)
    xs, ys = x[idx], y[idx]
    n, win = len(xs), max(3, int(frac * len(xs)))
    ysm = np.zeros(n)
    for i in range(n):
        d   = np.abs(xs - xs[i])
        nid = np.argsort(d)[:win]
        dm  = d[nid].max() + 1e-10
        w   = np.maximum((1-(d[nid]/dm)**3)**3, 0)
        xw, yw, ww = xs[nid], ys[nid], w
        denom = (ww*xw**2).sum() - (ww*xw).sum()**2/ww.sum()
        if abs(denom) < 1e-12:
            ysm[i] = (ww*yw).sum()/ww.sum()
        else:
            b1 = ((ww*xw*yw).sum()-(ww*xw).sum()*(ww*yw).sum()/ww.sum())/denom
            b0 = ((ww*yw).sum()-b1*(ww*xw).sum())/ww.sum()
            ysm[i] = b0+b1*xs[i]
    return xs, ysm

def poly_fit_with_ci(x, y, deg=3, ci=0.95, n_pts=200):
    xf = np.linspace(x.min(), x.max(), n_pts)
    yf = np.polyval(np.polyfit(x, y, deg), xf)
    bp = []
    for _ in range(200):
        ib = np.random.choice(len(x), len(x), replace=True)
        bp.append(np.polyval(np.polyfit(x[ib], y[ib], deg), xf))
    ba = np.array(bp)
    a  = 1 - ci
    return xf, yf, np.percentile(ba, a/2*100, axis=0), np.percentile(ba, (1-a/2)*100, axis=0)

# ====================== ★ v9核心：精确正负分区工具函数 ======================
def find_zero_crossings(x_curve, y_curve):
    """
    找到曲线与 y=0 的所有交叉点（线性插值精确定位）。
    返回：交叉点的 x 坐标列表
    """
    crossings = []
    for i in range(len(y_curve) - 1):
        y0, y1 = y_curve[i], y_curve[i+1]
        # 符号变化 → 存在零点
        if y0 * y1 < 0:
            # 线性插值求精确x坐标
            x0, x1 = x_curve[i], x_curve[i+1]
            x_cross = x0 + (-y0) * (x1 - x0) / (y1 - y0)
            crossings.append(float(x_cross))
    return crossings


def draw_precise_pos_neg_regions(ax, x_curve, y_curve, x_min, x_max, y_lo, y_hi):
    """
    根据Lowess曲线与SHAP=0的实际交叉点，精确绘制正负背景分区。
    正贡献区（curve>0）→ 浅红；负贡献区（curve<0）→ 浅蓝。
    返回：交叉点x坐标列表
    """
    # 找到所有交叉点
    crossings = find_zero_crossings(x_curve, y_curve)

    # 构建分段边界：[x_min] + crossings + [x_max]
    boundaries = [x_min] + sorted(crossings) + [x_max]

    for seg_idx in range(len(boundaries) - 1):
        x_lo_seg = boundaries[seg_idx]
        x_hi_seg = boundaries[seg_idx + 1]
        x_mid    = (x_lo_seg + x_hi_seg) / 2

        # 在该段中间位置查询Lowess曲线符号
        interp_y = np.interp(x_mid, x_curve, y_curve)

        if interp_y >= 0:
            # 正贡献区（浅红）
            ax.fill_betweenx(
                [y_lo, y_hi],
                x_lo_seg, x_hi_seg,
                color=CONFIG['pos_color'],
                alpha=CONFIG['pos_alpha'],
                zorder=0)
        else:
            # 负贡献区（浅蓝）
            ax.fill_betweenx(
                [y_lo, y_hi],
                x_lo_seg, x_hi_seg,
                color=CONFIG['neg_color'],
                alpha=CONFIG['neg_alpha'],
                zorder=0)

    return crossings


def annotate_crossings(ax, crossings, x_curve, y_curve, y_lo, y_hi):
    """
    在交叉点处绘制：
    - 红色垂直虚线
    - 红色圆点（标注在SHAP=0线上）
    - 特征值数值标注（对应图2/3/4风格）
    """
    for xc in crossings:
        # 垂直虚线
        ax.axvline(xc, color=CONFIG['cross_color'],
                   ls='--', lw=1.8, alpha=0.9, zorder=6)
        # SHAP=0 处的圆点
        ax.scatter([xc], [0],
                   color=CONFIG['cross_color'],
                   s=60, zorder=8, edgecolors='white', linewidth=1.2)
        # 数值标注（红色加粗）
        ax.text(xc, y_lo + (y_hi - y_lo) * 0.06,
                f'{xc:.2f}',
                ha='center', va='bottom',
                fontsize=10, fontweight='bold',
                color=CONFIG['cross_color'],
                zorder=9,
                bbox=dict(boxstyle='round,pad=0.2',
                          fc='white', ec=CONFIG['cross_color'],
                          alpha=0.85, lw=1.2))


def gp_predict_scaled(model, scaler, X_raw):
    return model.predict(scaler.transform(X_raw))

def load_gp_model_and_data():
    print("=== 加载TOP1高斯过程模型和数据 ===")
    with open(CONFIG['pkl_file'], 'rb') as f:
        md = pickle.load(f)
    gp_model, scaler, feature_names = md['model'], md['scaler'], md['features']
    print(f"✓ 模型: {md['model_name']}  特征: {feature_names}")
    df       = pd.read_excel(CONFIG['data_file'])
    df_valid = df[feature_names+[CONFIG['target_col']]].dropna()
    df_valid = df_valid[df_valid[CONFIG['target_col']]>0].reset_index(drop=True)
    X_all    = df_valid[feature_names].values
    y_all    = df_valid[CONFIG['target_col']].values
    print(f"✓ 有效样本: {len(df_valid)}")
    return {'model_name': md['model_name'], 'model_instance': gp_model,
            'scaler': scaler, 'features': feature_names,
            'X_data': X_all, 'y_data': y_all,
            'test_r2': md.get('test_r2_true', np.nan)}

def build_shap_explainer(model_info):
    print("\n  构建 SHAP KernelExplainer...")
    model, scaler, X_data = (model_info['model_instance'],
                              model_info['scaler'], model_info['X_data'])
    def predict_fn(X): return gp_predict_scaled(model, scaler, X)
    n_bg = min(CONFIG['shap_background_samples'], X_data.shape[0])
    bg   = shap.kmeans(X_data, n_bg)
    exp  = shap.KernelExplainer(predict_fn, bg)
    print(f"    ✓ 构建完成（背景{n_bg}点）")
    return exp

def compute_shap_values(explainer, model_info):
    print("\n  计算 SHAP 值...")
    X_data = model_info['X_data']
    n      = (X_data.shape[0] if CONFIG['shap_explain_samples'] is None
              else min(CONFIG['shap_explain_samples'], X_data.shape[0]))
    X_exp  = X_data[:n]
    sv     = explainer.shap_values(X_exp, nsamples='auto')
    print(f"    ✓ 完成，shape: {np.array(sv).shape}")
    return sv, X_exp

# ==============================================================
# 1. Summary Plot
# ==============================================================
def plot_shap_summary(shap_values, X_explain, features, out_dir):
    print("\n  生成 SHAP Summary Plot...")
    base     = os.path.join(out_dir, 'shap_summary')
    sv       = np.array(shap_values)
    n_s, n_f = sv.shape
    mean_abs = np.mean(np.abs(sv), axis=0)
    rank_idx = np.argsort(mean_abs)[::-1]

    for plot_type, fname, title in [
        ('bar',    'SHAP_Bar_Importance.png', 'SHAP 特征重要性（平均|SHAP值|）'),
        ('dot',    'SHAP_Beeswarm.png',       'SHAP Beeswarm Plot'),
        ('violin', 'SHAP_Violin.png',         'SHAP Violin Plot'),
    ]:
        fig, ax = plt.subplots(figsize=CONFIG['figure_size'])
        shap.summary_plot(shap_values, X_explain,
                          feature_names=features, plot_type=plot_type, show=False)
        plt.title(title, fontsize=14, fontweight='bold')
        plt.tight_layout()
        plt.savefig(os.path.join(base, fname),
                    dpi=CONFIG['dpi'], bbox_inches='tight'); plt.close()

    cols_p = min(3, n_f); rows_p = (n_f+cols_p-1)//cols_p
    fig, axes_arr = plt.subplots(rows_p, cols_p, figsize=(6*cols_p, 5*rows_p))
    if n_f == 1: axes_arr = np.array([[axes_arr]])
    elif rows_p == 1: axes_arr = axes_arr.reshape(1, -1)
    for idx in range(n_f):
        r_i, c_i = divmod(idx, cols_p)
        ax, fi   = axes_arr[r_i, c_i], rank_idx[idx]
        sc = ax.scatter(X_explain[:, fi], sv[:, fi],
                        c=X_explain[:, fi], cmap='coolwarm', alpha=0.6, s=30)
        plt.colorbar(sc, ax=ax)
        ax.axhline(0, color='k', ls='--', alpha=0.4)
        z  = np.polyfit(X_explain[:, fi], sv[:, fi], 1)
        xs = np.linspace(X_explain[:, fi].min(), X_explain[:, fi].max(), 100)
        ax.plot(xs, np.polyval(z, xs), 'k-', lw=1.5, alpha=0.7)
        ax.set_xlabel(features[fi], fontsize=9); ax.set_ylabel('SHAP值', fontsize=9)
        ax.set_title(f'#{idx+1} {features[fi]}\n|SHAP|均值={mean_abs[fi]:.4f}',
                     fontsize=9, fontweight='bold'); ax.grid(alpha=0.3)
    for idx in range(n_f, rows_p*cols_p):
        r_i, c_i = divmod(idx, cols_p); axes_arr[r_i, c_i].axis('off')
    plt.suptitle('SHAP 各特征散点汇总（按重要性排序）', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(base, 'SHAP_All_Features_Comprehensive.png'),
                dpi=CONFIG['dpi'], bbox_inches='tight'); plt.close()
    print("    ✓ 4张图已保存")

    df_imp = pd.DataFrame({
        'Feature': features, 'Mean_Abs_SHAP': mean_abs,
        'Rank': pd.Series(mean_abs).rank(ascending=False).astype(int).values,
    }).sort_values('Mean_Abs_SHAP', ascending=False).reset_index(drop=True)
    df_shap_wide = pd.DataFrame(sv, columns=[f'SHAP_{safe_filename(f)}' for f in features])
    df_shap_wide.insert(0, 'Sample_Index', range(n_s))
    df_feat_wide = pd.DataFrame(X_explain, columns=[f'X_{safe_filename(f)}' for f in features])
    df_feat_wide.insert(0, 'Sample_Index', range(n_s))
    df_pred_wide = pd.DataFrame({'Sample_Index': range(n_s), 'Sum_SHAP': sv.sum(axis=1)})
    for fi, fn in enumerate(features):
        df_pred_wide[f'SHAP_{safe_filename(fn)}'] = sv[:, fi]
    stat_names = ['Mean','Std','Min','Q1','Median','Q3','Max','Mean_Abs','Skewness']
    stat_data  = {}
    for fi, fn in enumerate(features):
        col = sv[:, fi]
        stat_data[fn] = [col.mean(), col.std(), col.min(),
                         np.percentile(col,25), np.median(col),
                         np.percentile(col,75), col.max(),
                         np.mean(np.abs(col)), float(pd.Series(col).skew())]
    df_violin = pd.DataFrame(stat_data, index=stat_names)
    df_violin.insert(0, 'Statistic', stat_names); df_violin = df_violin.reset_index(drop=True)
    write_excel_multisheets(
        os.path.join(base, 'SHAP_Summary_Data.xlsx'),
        {'Feature_Importance': df_imp, 'SHAP_Values_Matrix': df_shap_wide,
         'Feature_Values_Matrix': df_feat_wide, 'SHAP_by_Sample': df_pred_wide,
         'Violin_Stats': df_violin},
        {'n_samples': n_s, 'n_features': n_f, 'Format': 'Wide Table'})
    print("    ✓ SHAP_Summary_Data.xlsx（5 Sheets）→ shap_summary/")

# ==============================================================
# 2. 逐样本 SHAP 热图
# ==============================================================
def plot_shap_sample_heatmap(shap_values, X_explain, features,
                              explainer, y_data, out_dir):
    print("\n  生成 逐样本SHAP热图...")
    base       = os.path.join(out_dir, 'shap_heatmap')
    sv         = np.array(shap_values)
    base_value = explainer.expected_value
    pred_vals  = sv.sum(axis=1) + base_value
    mean_abs   = np.mean(np.abs(sv), axis=0)
    feat_order = np.argsort(mean_abs)[::-1]
    sv_ordered = sv[:, feat_order]
    feat_names_ordered = [features[i] for i in feat_order]

    sort_mode = CONFIG['heatmap_sort_by']
    if sort_mode == 'prediction':
        sample_order = np.argsort(pred_vals)[::-1]; sort_label = 'sorted by predicted KQ (↓)'
    elif sort_mode == 'cluster':
        from scipy.cluster.hierarchy import linkage, leaves_list
        from scipy.spatial.distance  import pdist
        sample_order = leaves_list(linkage(pdist(sv,'euclidean'),'ward'))
        sort_label   = 'hierarchical clustering'
    else:
        sample_order = np.arange(len(pred_vals)); sort_label = 'original index'

    sv_plot   = sv_ordered[sample_order]
    pred_plot = pred_vals[sample_order]
    n_s, n_f  = sv_plot.shape
    vmax = np.percentile(np.abs(sv_plot), 98); vmin = -vmax

    fig = plt.figure(figsize=(3*n_f+2, max(10, n_s*0.06+3)))
    gs  = fig.add_gridspec(1, n_f+1, width_ratios=[1]*n_f+[0.08], hspace=0.05, wspace=0.03)
    axes_feat = [fig.add_subplot(gs[0, k]) for k in range(n_f)]
    ax_cbar   = fig.add_subplot(gs[0, n_f])
    for k, (ax, fn) in enumerate(zip(axes_feat, feat_names_ordered)):
        im = ax.imshow(sv_plot[:, k].reshape(-1,1), aspect='auto',
                       cmap=CONFIG['heatmap_cmap'], vmin=vmin, vmax=vmax, interpolation='nearest')
        ax.set_xticks([0]); ax.set_xticklabels([fn], rotation=45, ha='right', fontsize=10, fontweight='bold')
        ax.set_yticks([])
        if k == 0:
            step = max(1, n_s//15); ax.set_yticks(list(range(0, n_s, step)))
            ax.set_yticklabels([str(sample_order[p]) for p in range(0,n_s,step)], fontsize=7)
            ax.set_ylabel('number of sample', fontsize=11, fontweight='bold')
    plt.colorbar(im, cax=ax_cbar, label='SHAP values')
    fig.suptitle(f'SHAP Values per Sample  |  GP Matérn-0.5\n({sort_label}, n={n_s})',
                 fontsize=13, fontweight='bold', y=1.01)
    plt.savefig(os.path.join(base, 'SHAP_Sample_Heatmap_Main.png'),
                dpi=CONFIG['dpi'], bbox_inches='tight'); plt.close()

    fig = plt.figure(figsize=(3*n_f+4, max(10, n_s*0.06+4)))
    gs2 = fig.add_gridspec(2, n_f+2, height_ratios=[0.08,1],
                           width_ratios=[1]*n_f+[0.06,0.35], hspace=0.04, wspace=0.03)
    ax_top = fig.add_subplot(gs2[0, :n_f])
    ax_top.bar(range(n_f), mean_abs[feat_order],
               color=plt.cm.YlOrRd(mean_abs[feat_order]/mean_abs[feat_order].max()),
               edgecolor='white', linewidth=0.5)
    ax_top.set_xlim(-0.5, n_f-0.5); ax_top.set_xticks([]); ax_top.set_ylabel('Mean|SHAP|', fontsize=8)
    ax_top.set_title('SHAP Sample Heatmap with Feature Importance & Prediction', fontsize=12, fontweight='bold')
    ax_top.grid(alpha=0.3, axis='y')
    axes_f2 = [fig.add_subplot(gs2[1, k]) for k in range(n_f)]
    for k, (ax, fn) in enumerate(zip(axes_f2, feat_names_ordered)):
        im2 = ax.imshow(sv_plot[:, k].reshape(-1,1), aspect='auto',
                        cmap=CONFIG['heatmap_cmap'], vmin=vmin, vmax=vmax, interpolation='nearest')
        ax.set_xticks([0]); ax.set_xticklabels([fn], rotation=45, ha='right', fontsize=9); ax.set_yticks([])
        if k == 0:
            step = max(1, n_s//15); ax.set_yticks(list(range(0,n_s,step)))
            ax.set_yticklabels([str(sample_order[p]) for p in range(0,n_s,step)], fontsize=7)
            ax.set_ylabel('number of sample', fontsize=11, fontweight='bold')
    plt.colorbar(im2, cax=fig.add_subplot(gs2[1, n_f]), label='SHAP values')
    ax_pred = fig.add_subplot(gs2[1, n_f+1])
    im_pred = ax_pred.imshow(pred_plot.reshape(-1,1), aspect='auto', cmap='RdYlGn',
                              vmin=pred_plot.min(), vmax=pred_plot.max(), interpolation='nearest')
    ax_pred.set_xticks([0]); ax_pred.set_xticklabels(['Predicted\nKQ'], fontsize=9); ax_pred.set_yticks([])
    step2 = max(1, n_s//8)
    for tp in range(0, n_s, step2):
        ax_pred.text(1.05, tp, f'{pred_plot[tp]:.2f}', va='center', fontsize=7,
                     transform=ax_pred.get_yaxis_transform())
    plt.colorbar(im_pred, ax=ax_pred, location='right', label='KQ', pad=0.8, shrink=0.6)
    plt.savefig(os.path.join(base, 'SHAP_Sample_Heatmap_Full.png'),
                dpi=CONFIG['dpi'], bbox_inches='tight'); plt.close()

    fig, axes = plt.subplots(1, n_f, figsize=(2.5*n_f, max(8, n_s*0.05+2)))
    for k, (ax, fn) in enumerate(zip(axes, feat_names_ordered)):
        col = sv_plot[:, k]; vmk = np.percentile(np.abs(col), 98)
        imk = ax.imshow(col.reshape(-1,1), aspect='auto', cmap=CONFIG['heatmap_cmap'],
                        vmin=-vmk, vmax=vmk, interpolation='nearest')
        ax.set_xticks([0]); ax.set_xticklabels([fn], rotation=45, ha='right', fontsize=9); ax.set_yticks([])
        plt.colorbar(imk, ax=ax, shrink=0.4, pad=0.02)
        if k == 0:
            step = max(1, n_s//12); ax.set_yticks(list(range(0,n_s,step)))
            ax.set_yticklabels([str(sample_order[p]) for p in range(0,n_s,step)], fontsize=7)
            ax.set_ylabel('number of sample', fontsize=10)
    plt.suptitle('SHAP Values per Feature (Independent Color Scale)', fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(base, 'SHAP_Sample_Heatmap_PerFeature.png'),
                dpi=CONFIG['dpi'], bbox_inches='tight'); plt.close()
    print("    ✓ 3张热图已保存")

    df_shap_w = pd.DataFrame(sv_plot, columns=[f'SHAP_{safe_filename(fn)}' for fn in feat_names_ordered])
    df_shap_w.insert(0, 'Sort_Rank', range(1, n_s+1)); df_shap_w.insert(1, 'Original_Sample_Index', sample_order)
    df_shap_w.insert(2, 'Predicted_KQ', pred_plot)
    feat_vals_sorted = X_explain[sample_order]
    df_feat_w = pd.DataFrame(feat_vals_sorted, columns=[f'X_{safe_filename(fn)}' for fn in feat_names_ordered])
    df_feat_w.insert(0, 'Sort_Rank', range(1, n_s+1)); df_feat_w.insert(1, 'Original_Sample_Index', sample_order)
    df_feat_w.insert(2, 'Predicted_KQ', pred_plot)
    df_feat_ord = pd.DataFrame({'Importance_Rank': range(1, n_f+1), 'Feature': feat_names_ordered,
                                'Mean_Abs_SHAP': mean_abs[feat_order], 'Original_Index': feat_order})
    write_excel_multisheets(
        os.path.join(base, 'SHAP_Heatmap_Data.xlsx'),
        {'Heatmap_SHAP_Wide': df_shap_w, 'Feature_Values_Wide': df_feat_w, 'Feature_Order': df_feat_ord},
        {'sort_mode': sort_mode, 'n_samples': n_s, 'n_features': n_f,
         'vmax': float(vmax), 'baseline': float(base_value), 'Format': 'Wide Table'})
    print("    ✓ SHAP_Heatmap_Data.xlsx（3 Sheets）→ shap_heatmap/")

# ==============================================================
# 3. 原版单特征依赖图
# ==============================================================
def plot_shap_dependence(shap_values, X_explain, features, out_dir):
    print("\n  生成 SHAP 单特征依赖图（原版）...")
    base     = os.path.join(out_dir, 'shap_dependence')
    sv       = np.array(shap_values)
    mean_abs = np.mean(np.abs(sv), axis=0)
    rank_idx = np.argsort(mean_abs)[::-1]

    for i, feat in enumerate(features):
        print(f"    处理: {feat}")
        try:
            color_feat = rank_idx[0] if rank_idx[0] != i else rank_idx[1]
            fig, axes  = plt.subplots(1, 2, figsize=(16, 6))
            shap.dependence_plot(i, shap_values, X_explain, feature_names=features,
                                 interaction_index=color_feat, ax=axes[0], show=False)
            axes[0].set_title(f'SHAP依赖图: {feat}\n（着色: {features[color_feat]}）',
                              fontsize=11, fontweight='bold'); axes[0].grid(alpha=0.3)
            x_v, s_v = X_explain[:, i], sv[:, i]
            sc = axes[1].scatter(x_v, s_v, c=X_explain[:, color_feat],
                                 cmap='coolwarm', alpha=0.7, s=40)
            plt.colorbar(sc, ax=axes[1], label=features[color_feat])
            z  = np.polyfit(x_v, s_v, 2); xs = np.linspace(x_v.min(), x_v.max(), 200)
            axes[1].plot(xs, np.polyval(z, xs), 'k--', lw=2, label='二次拟合')
            axes[1].axhline(0, color='gray', ls='--', alpha=0.5)
            axes[1].set_xlabel(feat, fontsize=11, fontweight='bold')
            axes[1].set_ylabel('SHAP值', fontsize=11, fontweight='bold')
            axes[1].set_title(f'特征值 vs SHAP值\nr={np.corrcoef(x_v,s_v)[0,1]:.4f}',
                              fontsize=11, fontweight='bold')
            axes[1].legend(); axes[1].grid(alpha=0.3)
            axes[1].text(0.02, 0.98,
                         f'SHAP均值: {s_v.mean():.4f}\nSHAP std: {s_v.std():.4f}\n'
                         f'平均|SHAP|: {np.mean(np.abs(s_v)):.4f}\n'
                         f'排名: #{np.where(rank_idx==i)[0][0]+1}',
                         transform=axes[1].transAxes,
                         bbox=dict(boxstyle='round,pad=0.3', fc='lightcyan', alpha=0.8),
                         va='top', fontsize=9)
            plt.tight_layout()
            sname = safe_filename(feat)
            plt.savefig(os.path.join(base, f'SHAP_Dependence_{sname}.png'),
                        dpi=CONFIG['dpi'], bbox_inches='tight'); plt.close()
            cfname = safe_filename(features[color_feat])
            df_raw = pd.DataFrame({'Sample_Index': range(len(x_v)),
                                   f'X_{sname}': x_v, f'SHAP_{sname}': s_v,
                                   f'Color_{cfname}': X_explain[:, color_feat]})
            df_poly = pd.DataFrame({f'X_{sname}_Fit': xs, f'SHAP_{sname}_Poly2': np.polyval(z, xs)})
            write_excel_multisheets(
                os.path.join(base, f'SHAP_Dependence_{sname}.xlsx'),
                {'Raw_Data': df_raw, 'Poly_Fit': df_poly},
                {'Feature': feat, 'Color_Feature': features[color_feat],
                 'Pearson_r': float(np.corrcoef(x_v, s_v)[0,1]),
                 'Rank': int(np.where(rank_idx==i)[0][0]+1), 'Format': 'Wide Table'})
            print(f"      ✓ {feat} → shap_dependence/")
        except Exception as e:
            print(f"      ❌ {feat} 失败: {e}"); plt.close('all')

# ==============================================================
# ★ v9核心：_draw_enhanced_dep（精确正负分区 + 交叉点标注）
# ==============================================================
def _draw_enhanced_dep(ax, x_v, s_v, color_vals, feat_name, colorbar_label,
                        rank, n_total, mean_abs_shap):
    """
    ★ v9修改：
    1. 先计算Lowess曲线
    2. 找到曲线与SHAP=0的精确交叉点
    3. 按交叉点精确划分正负背景区域
    4. 在交叉点处绘制红色虚线 + 红点 + 数值标注
    """
    # ── 先计算Lowess（需要在绘图前就知道曲线走向）─────────────
    try:
        xlw, ylw = lowess_smooth(x_v, s_v, frac=CONFIG['dependence_lowess_frac'])
    except Exception:
        xlw, ylw = np.sort(x_v), np.zeros_like(x_v)

    # ── 计算Y轴范围（含padding）────────────────────────────────
    x_min = float(x_v.min())
    x_max = float(x_v.max())
    y_pad = (float(s_v.max()) - float(s_v.min())) * 0.08
    y_lo  = float(s_v.min()) - y_pad
    y_hi  = float(s_v.max()) + y_pad

    # ── ① 精确正负背景分区（★ v9核心）────────────────────────
    crossings = draw_precise_pos_neg_regions(ax, xlw, ylw, x_min, x_max, y_lo, y_hi)

    # ── ② SHAP=0 分界线 ───────────────────────────────────────
    ax.axhline(0, color='gray', ls='-', lw=1.0, alpha=0.7, zorder=2)

    # ── ③ 散点（颜色=自身特征值）──────────────────────────────
    sc = ax.scatter(x_v, s_v,
                    c=color_vals,
                    cmap='coolwarm',
                    vmin=np.percentile(x_v, 5),
                    vmax=np.percentile(x_v, 95),
                    alpha=0.65, s=35,
                    edgecolors='none',
                    zorder=3,
                    label=f'Samples (n={len(x_v)})')
    cb = plt.colorbar(sc, ax=ax, pad=0.01)
    cb.set_label(colorbar_label, fontsize=8, rotation=270, labelpad=12)
    cb.ax.tick_params(labelsize=7)

    # ── ④ Lowess 平滑拟合线（已算好，直接画）─────────────────
    ax.plot(xlw, ylw, '-', color='#d62728', lw=2.5, zorder=5, label='Lowess Fit')

    # ── ⑤ 多项式拟合 + Bootstrap置信带 ──────────────────────
    try:
        xf, yf, ylo, yhi = poly_fit_with_ci(
            x_v, s_v, deg=CONFIG['dependence_poly_degree'],
            ci=CONFIG['dependence_ci_alpha'])
        ax.fill_between(xf, ylo, yhi, alpha=0.18, color='gray', zorder=1,
                        label=f'{int(CONFIG["dependence_ci_alpha"]*100)}% CI')
        ax.plot(xf, yf, '--', color='#1f77b4', lw=2.0, zorder=4, label='Polynomial Fit')
    except Exception:
        pass

    # ── ⑥ 交叉点标注（★ v9核心：红线+红点+数值）──────────────
    if crossings:
        annotate_crossings(ax, crossings, xlw, ylw, y_lo, y_hi)

    # ── ⑦ 统计标注（右上角）──────────────────────────────────
    r_val, _ = pearsonr(x_v, s_v)
    ax.text(0.97, 0.97,
            f'Mean|SHAP|={mean_abs_shap:.4f}\n'
            f'Pearson r={r_val:.3f}\n'
            f'Rank: #{rank}/{n_total}',
            transform=ax.transAxes, fontsize=8, va='top', ha='right',
            bbox=dict(boxstyle='round,pad=0.3', fc='lightyellow', alpha=0.85, ec='gray'),
            zorder=9)

    ax.set_xlabel(feat_name, fontsize=11, fontweight='bold')
    ax.set_ylabel('SHAP Value', fontsize=11, fontweight='bold')
    ax.set_title(f'SHAP Dependence Analysis: {feat_name}', fontsize=11, fontweight='bold')

    # 图例
    handles, labels = ax.get_legend_handles_labels()
    ax.legend(handles, labels, fontsize=8, loc='upper left', framealpha=0.85)
    ax.grid(alpha=0.2, zorder=0)
    ax.set_ylim(y_lo, y_hi)
    ax.set_xlim(x_min - (x_max - x_min) * 0.02,
                x_max + (x_max - x_min) * 0.02)

# ==============================================================
# 4. 增强版 SHAP Dependence 图
# ==============================================================
def plot_shap_dependence_enhanced(shap_values, X_explain, features,
                                   explainer, y_data, out_dir):
    print("\n  生成 增强版SHAP Dependence图（v9: 精确Positive/Negative分区 + 交叉点标注）...")
    base     = os.path.join(out_dir, 'shap_dependence_enhanced')
    sv       = np.array(shap_values)
    mean_abs = np.mean(np.abs(sv), axis=0)
    rank_idx = np.argsort(mean_abs)[::-1]
    n_f      = len(features)

    # ── 合并大图 ──
    cols = min(3, n_f); rows = (n_f+cols-1)//cols
    fig_all, axes_all = plt.subplots(rows, cols, figsize=(7*cols, 6*rows))
    if n_f == 1: axes_all = np.array([[axes_all]])
    elif rows == 1: axes_all = axes_all.reshape(1, -1)

    for plot_idx, fi in enumerate(rank_idx):
        r_i, c_i = divmod(plot_idx, cols)
        x_v = X_explain[:, fi].astype(float)
        s_v = sv[:, fi].astype(float)
        _draw_enhanced_dep(axes_all[r_i, c_i],
                           x_v, s_v, x_v,
                           features[fi], features[fi],
                           plot_idx+1, n_f, mean_abs[fi])

    for pi in range(n_f, rows*cols):
        r_i, c_i = divmod(pi, cols); axes_all[r_i, c_i].axis('off')
    plt.suptitle(
        'SHAP Dependence Analysis — Enhanced\n'
        '(Lowess + Poly CI | Precise Positive/Negative Regions + Crossover Annotations)',
        fontsize=14, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.savefig(os.path.join(base, 'SHAP_Dependence_Enhanced_All.png'),
                dpi=CONFIG['dpi'], bbox_inches='tight'); plt.close()
    print("    ✓ 合并大图已保存")

    summary_raw_sheets = {}

    for fi in rank_idx:
        feat     = features[fi]
        x_v      = X_explain[:, fi].astype(float)
        s_v      = sv[:, fi].astype(float)
        rank_pos = int(np.where(rank_idx==fi)[0][0])+1
        sname    = safe_filename(feat)

        fig_s, ax_s = plt.subplots(figsize=(9, 6))
        _draw_enhanced_dep(ax_s, x_v, s_v, x_v,
                           feat, feat, rank_pos, n_f, mean_abs[fi])
        plt.tight_layout()
        plt.savefig(os.path.join(base, f'SHAP_Dependence_Enhanced_{sname}.png'),
                    dpi=CONFIG['dpi'], bbox_inches='tight'); plt.close()

        # ── Excel 宽表（4个Sheet）──
        df_raw = pd.DataFrame({
            'Sample_Index':   range(len(x_v)),
            f'X_{sname}':     x_v,
            f'SHAP_{sname}':  s_v,
            f'Color_{sname}': x_v,
        })
        summary_raw_sheets[sname[:31]] = df_raw

        try:
            xlw, ylw = lowess_smooth(x_v, s_v, frac=CONFIG['dependence_lowess_frac'])
            df_lowess = pd.DataFrame({
                f'X_{sname}_Lowess':    xlw,
                f'SHAP_{sname}_Lowess': ylw,
            })
            # 计算交叉点（用于Excel记录）
            crossings_excel = find_zero_crossings(xlw, ylw)
        except Exception:
            df_lowess = pd.DataFrame({f'X_{sname}_Lowess': [], f'SHAP_{sname}_Lowess': []})
            crossings_excel = []

        try:
            xf, yf, ylo, yhi = poly_fit_with_ci(
                x_v, s_v, deg=CONFIG['dependence_poly_degree'],
                ci=CONFIG['dependence_ci_alpha'])
            df_poly_ci = pd.DataFrame({
                f'X_{sname}_Fit':      xf,
                f'SHAP_{sname}_Poly':  yf,
                f'SHAP_{sname}_CI_Lo': ylo,
                f'SHAP_{sname}_CI_Hi': yhi,
            })
        except Exception:
            df_poly_ci = pd.DataFrame()

        # ★ v9：Sheet4 = 精确分区信息（交叉点坐标 + 各区间符号）
        r_val, _ = pearsonr(x_v, s_v)
        x_min_f, x_max_f = float(x_v.min()), float(x_v.max())
        boundaries = [x_min_f] + sorted(crossings_excel) + [x_max_f]
        region_rows = []
        for seg_idx in range(len(boundaries) - 1):
            x_lo_seg = boundaries[seg_idx]
            x_hi_seg = boundaries[seg_idx + 1]
            x_mid    = (x_lo_seg + x_hi_seg) / 2
            if len(xlw) > 0:
                interp_y = float(np.interp(x_mid, xlw, ylw))
            else:
                interp_y = 0.0
            region_rows.append({
                'Segment':          seg_idx + 1,
                'X_Start':          x_lo_seg,
                'X_End':            x_hi_seg,
                'Region':           'Positive' if interp_y >= 0 else 'Negative',
                'Lowess_Y_at_Mid':  interp_y,
                'Background_Color': CONFIG['pos_color'] if interp_y >= 0 else CONFIG['neg_color'],
            })

        df_regions = pd.DataFrame(region_rows)
        df_crossings = pd.DataFrame({
            'Crossing_Index': range(1, len(crossings_excel)+1),
            'X_Crossing':     crossings_excel,
            'SHAP_at_Cross':  0.0,
            'Feature':        feat,
            'Pearson_r':      r_val,
            'Mean_Abs_SHAP':  float(mean_abs[fi]),
            'Importance_Rank': rank_pos,
        }) if crossings_excel else pd.DataFrame({
            'Note': [f'No crossing found for {feat}']})

        # 合并分区信息和交叉点到同一Sheet
        df_regions_combined = pd.concat([
            df_regions,
            pd.DataFrame([{'Segment': '---', 'X_Start': '---', 'X_End': '---',
                           'Region': '=== Crossings ===',
                           'Lowess_Y_at_Mid': '---', 'Background_Color': '---'}]),
            df_crossings.rename(columns={'Crossing_Index': 'Segment',
                                          'X_Crossing': 'X_Start'})
        ], ignore_index=True)

        write_excel_multisheets(
            os.path.join(base, f'SHAP_Dependence_Enhanced_{sname}.xlsx'),
            {
                'Raw_Data':              df_raw,
                'Lowess_Fit':            df_lowess,
                'Poly_Fit_CI':           df_poly_ci,
                'Precise_Regions':       df_regions,    # ★ v9
                'Zero_Crossings':        df_crossings,  # ★ v9
            },
            {'Feature':         feat,
             'Color_Source':    'Own feature value (X axis)',
             'Region_Method':   'Lowess curve zero crossings (precise)',  # ★ v9
             'N_Crossings':     len(crossings_excel),
             'Crossings_X':     str([round(c,3) for c in crossings_excel]),
             'Lowess_frac':     CONFIG['dependence_lowess_frac'],
             'Poly_degree':     CONFIG['dependence_poly_degree'],
             'CI_level':        CONFIG['dependence_ci_alpha'],
             'Format':          'Wide Table'})
        print(f"      ✓ {feat}（{len(crossings_excel)}个交叉点）→ shap_dependence_enhanced/")

    write_excel_multisheets(
        os.path.join(base, 'SHAP_Dependence_Enhanced_Summary.xlsx'),
        summary_raw_sheets,
        {'Color_Source':  'Own feature value',
         'Region_Method': 'Precise Positive/Negative via Lowess zero crossings (v9)'})
    print("    ✓ SHAP_Dependence_Enhanced_Summary.xlsx → shap_dependence_enhanced/")

# ==============================================================
# 5. SHAP 交互热力图
# ==============================================================
def plot_shap_interaction_heatmap(shap_values, X_explain, features,
                                   explainer, out_dir):
    print("\n  生成 SHAP 特征交互热力图...")
    base   = os.path.join(out_dir, 'shap_interaction')
    n_feat = len(features)
    sv     = np.array(shap_values)
    corr_mat  = np.corrcoef(sv.T)
    joint_mat = np.zeros((n_feat, n_feat))
    for ii in range(n_feat):
        for jj in range(n_feat):
            joint_mat[ii, jj] = (np.mean(np.abs(sv[:, ii])) if ii==jj
                                  else np.mean(np.abs(sv[:, ii])*np.abs(sv[:, jj])))

    fig, axes = plt.subplots(1, 2, figsize=(18, 7))
    im1 = axes[0].imshow(corr_mat, cmap='RdBu_r', vmin=-1, vmax=1)
    axes[0].set_xticks(range(n_feat)); axes[0].set_yticks(range(n_feat))
    axes[0].set_xticklabels(features, rotation=45, ha='right', fontsize=9)
    axes[0].set_yticklabels(features, fontsize=9)
    plt.colorbar(im1, ax=axes[0], label='Pearson相关系数')
    for ii in range(n_feat):
        for jj in range(n_feat):
            axes[0].text(jj, ii, f'{corr_mat[ii,jj]:.2f}', ha='center', va='center',
                         fontsize=8, color='white' if abs(corr_mat[ii,jj])>0.5 else 'black')
    axes[0].set_title('SHAP值相关性矩阵', fontsize=12, fontweight='bold')
    im2 = axes[1].imshow(joint_mat, cmap='YlOrRd')
    axes[1].set_xticks(range(n_feat)); axes[1].set_yticks(range(n_feat))
    axes[1].set_xticklabels(features, rotation=45, ha='right', fontsize=9)
    axes[1].set_yticklabels(features, fontsize=9)
    plt.colorbar(im2, ax=axes[1], label='联合SHAP强度')
    for ii in range(n_feat):
        for jj in range(n_feat):
            axes[1].text(jj, ii, f'{joint_mat[ii,jj]:.3f}', ha='center', va='center', fontsize=8)
    axes[1].set_title('SHAP联合重要性热力图', fontsize=12, fontweight='bold')
    plt.suptitle('SHAP特征交互分析', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(base, 'SHAP_Interaction_Heatmap.png'),
                dpi=CONFIG['dpi'], bbox_inches='tight'); plt.close()

    rows = []
    for ii in range(n_feat):
        for jj in range(ii+1, n_feat):
            rows.append({'Rank': len(rows)+1,
                         'Feature_Pair': f'{features[ii]} × {features[jj]}',
                         'Feature1': features[ii], 'Feature2': features[jj],
                         'SHAP_Correlation': corr_mat[ii,jj],
                         'Joint_SHAP_Strength': joint_mat[ii,jj]})
    df_int = pd.DataFrame(rows).sort_values('Joint_SHAP_Strength', ascending=False)
    df_int['Rank'] = range(1, len(df_int)+1)

    fig, ax = plt.subplots(figsize=(12, 6))
    names = df_int['Feature_Pair'].tolist(); strs = df_int['Joint_SHAP_Strength'].values
    bars  = ax.bar(range(len(names)), strs, color=plt.cm.YlOrRd(np.linspace(0.3,1.0,len(names))))
    ax.set_xticks(range(len(names))); ax.set_xticklabels(names, rotation=45, ha='right', fontsize=9)
    ax.set_xlabel('特征交互对', fontsize=12, fontweight='bold')
    ax.set_ylabel('联合SHAP强度', fontsize=12, fontweight='bold')
    ax.set_title('SHAP特征交互强度排序', fontsize=14, fontweight='bold')
    for bar, s in zip(bars, strs):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+max(strs)*0.01,
                f'{s:.4f}', ha='center', va='bottom', fontsize=9)
    ax.grid(alpha=0.3, axis='y'); plt.tight_layout()
    plt.savefig(os.path.join(base, 'SHAP_Interaction_Ranking.png'),
                dpi=CONFIG['dpi'], bbox_inches='tight'); plt.close()
    print("    ✓ 2张图已保存")

    df_corr  = pd.DataFrame(corr_mat,  columns=features); df_corr.insert(0,  'Feature', features)
    df_joint = pd.DataFrame(joint_mat, columns=features); df_joint.insert(0, 'Feature', features)
    df_shap_mat = pd.DataFrame(sv, columns=[f'SHAP_{safe_filename(f)}' for f in features])
    df_shap_mat.insert(0, 'Sample_Index', range(len(sv)))
    write_excel_multisheets(
        os.path.join(base, 'SHAP_Interaction_Data.xlsx'),
        {'Correlation_Matrix': df_corr, 'Joint_Matrix': df_joint,
         'Interaction_Ranking': df_int, 'SHAP_Values_Matrix': df_shap_mat},
        {'n_samples': len(sv), 'n_features': n_feat, 'Format': 'Wide Table Matrix'})
    print("    ✓ SHAP_Interaction_Data.xlsx（4 Sheets）→ shap_interaction/")

# ==============================================================
# 6. SHAP Force Plot
# ==============================================================
def plot_shap_force(shap_values, X_explain, features, explainer, y_data, out_dir):
    print("\n  生成 SHAP Force Plot...")
    base       = os.path.join(out_dir, 'shap_force')
    base_value = explainer.expected_value
    pred_vals  = np.sum(shap_values, axis=1) + base_value
    n_fp       = CONFIG['force_plot_samples']
    sorted_idx = np.argsort(pred_vals)
    selected   = np.unique(np.concatenate([
        sorted_idx[:n_fp//2],
        sorted_idx[len(sorted_idx)//2-1:len(sorted_idx)//2+1],
        sorted_idx[-(n_fp-n_fp//2):]
    ]))
    force_sheets = {}

    for sidx in selected:
        try:
            fig, ax  = plt.subplots(figsize=(14, 4))
            sv_row   = shap_values[sidx]; x_row = X_explain[sidx]; pred_val = pred_vals[sidx]
            pos_mask = sv_row >= 0; neg_mask = sv_row < 0
            all_items = (
                list(zip(sv_row[pos_mask],
                         [features[k] for k in range(len(features)) if pos_mask[k]],
                         [x_row[k]    for k in range(len(features)) if pos_mask[k]],
                         ['#d73027']*pos_mask.sum())) +
                list(zip(np.abs(sv_row[neg_mask]),
                         [features[k] for k in range(len(features)) if neg_mask[k]],
                         [x_row[k]    for k in range(len(features)) if neg_mask[k]],
                         ['#4575b4']*neg_mask.sum()))
            )
            all_items.sort(key=lambda x: x[0], reverse=True)
            vals_plot = [it[0]*(1 if it[3]=='#d73027' else -1) for it in all_items]
            labels    = [f'{it[1]}\n={it[2]:.3f}' for it in all_items]
            colors_fp = [it[3] for it in all_items]
            bars = ax.barh(range(len(vals_plot)), vals_plot, color=colors_fp, alpha=0.85, edgecolor='white')
            ax.set_yticks(range(len(labels))); ax.set_yticklabels(labels, fontsize=9)
            ax.axvline(0, color='k', lw=1.5)
            ax.set_xlabel('SHAP值', fontsize=11, fontweight='bold')
            ax.set_title(f'SHAP Force Plot — 样本#{sidx}\n基线: {base_value:.4f}  →  预测: {pred_val:.4f}',
                         fontsize=12, fontweight='bold'); ax.grid(alpha=0.3, axis='x')
            for bar, v in zip(bars, vals_plot):
                ax.text(v+(0.002 if v>=0 else -0.002), bar.get_y()+bar.get_height()/2,
                        f'{v:+.4f}', ha='left' if v>=0 else 'right', va='center', fontsize=8)
            plt.tight_layout()
            plt.savefig(os.path.join(base, f'SHAP_Force_Sample{sidx:03d}.png'),
                        dpi=CONFIG['dpi'], bbox_inches='tight'); plt.close()
            print(f"      ✓ 样本#{sidx} (预测={pred_val:.4f})")
            cumulative = []; c = base_value
            for v in vals_plot:
                c += v; cumulative.append(c)
            force_sheets[f'Force_Samp{sidx:03d}'] = pd.DataFrame({
                'Feature': [it[1] for it in all_items],
                'Feature_Value': [it[2] for it in all_items],
                'SHAP_Value': vals_plot, 'Abs_SHAP': [it[0] for it in all_items],
                'Direction': ['Pos' if it[3]=='#d73027' else 'Neg' for it in all_items],
                'Cumulative_KQ': cumulative, 'Baseline': base_value,
                'Predicted_KQ': pred_val, 'Sample_Index': sidx,
            })
        except Exception as e:
            print(f"      ❌ 样本#{sidx} 失败: {e}"); plt.close('all')

    df_shap_w = pd.DataFrame(shap_values, columns=[f'SHAP_{safe_filename(f)}' for f in features])
    df_shap_w.insert(0, 'Sample_Index', range(len(shap_values)))
    df_shap_w.insert(1, 'Predicted_KQ', pred_vals); df_shap_w.insert(2, 'Baseline', base_value)
    df_feat_w = pd.DataFrame(X_explain, columns=[f'X_{safe_filename(f)}' for f in features])
    df_feat_w.insert(0, 'Sample_Index', range(len(X_explain))); df_feat_w.insert(1, 'Predicted_KQ', pred_vals)
    all_sheets = {'SHAP_Values_Wide': df_shap_w, 'Feature_Values_Wide': df_feat_w}
    all_sheets.update(force_sheets)
    write_excel_multisheets(
        os.path.join(base, 'SHAP_Force_Data.xlsx'), all_sheets,
        {'n_samples': len(shap_values), 'n_features': len(features),
         'baseline': float(base_value), 'n_force_samples': len(selected), 'Format': 'Wide Table'})
    print(f"    ✓ SHAP_Force_Data.xlsx（{len(all_sheets)} Sheets）→ shap_force/")

# ==============================================================
# 7. SHAP Waterfall Plot
# ==============================================================
def plot_shap_waterfall(shap_values, X_explain, features, explainer, y_data, out_dir):
    print("\n  生成 SHAP Waterfall Plot...")
    base       = os.path.join(out_dir, 'shap_waterfall')
    base_value = explainer.expected_value
    pred_vals  = np.sum(shap_values, axis=1) + base_value
    sorted_idx = np.argsort(pred_vals)
    sample_map = {'High': sorted_idx[-1], 'Median': sorted_idx[len(sorted_idx)//2], 'Low': sorted_idx[0]}
    wf_sheets  = {}

    for label, sidx in sample_map.items():
        try:
            sv_row   = shap_values[sidx]; x_row = X_explain[sidx]; pred_val = pred_vals[sidx]
            order    = np.argsort(np.abs(sv_row))[::-1]
            sv_ord   = sv_row[order]; fn_ord = [features[k] for k in order]; fv_ord = [x_row[k] for k in order]
            fig, ax  = plt.subplots(figsize=(10, 8))
            cumsum   = base_value; lefts, widths, colors_wf = [], [], []
            for sv in sv_ord:
                lefts.append(min(cumsum, cumsum+sv)); widths.append(abs(sv))
                colors_wf.append('#d73027' if sv>=0 else '#4575b4'); cumsum+=sv
            ax.barh(list(range(len(sv_ord))), widths, left=lefts,
                    color=colors_wf, alpha=0.85, edgecolor='white', height=0.6)
            running = base_value
            for k, sv in enumerate(sv_ord):
                ax.plot([running,running],[k-0.35,k+0.35],'k-',lw=1,alpha=0.4); running+=sv
            ax.axvline(base_value, color='gray', ls='--', lw=1.5, label=f'基线={base_value:.4f}')
            ax.axvline(pred_val, color='black', ls='-', lw=2, label=f'预测值={pred_val:.4f}')
            ax.set_yticks(list(range(len(sv_ord))))
            ax.set_yticklabels([f'{fn}\n= {fv:.3f}' for fn,fv in zip(fn_ord,fv_ord)], fontsize=9)
            ax.set_xlabel('预测值', fontsize=11, fontweight='bold')
            ax.set_title(f'SHAP Waterfall — {label} (#{sidx})\n基线: {base_value:.4f}  →  预测: {pred_val:.4f}',
                         fontsize=12, fontweight='bold')
            ax.legend(fontsize=10); ax.grid(alpha=0.3, axis='x')
            for k,(sv,left,w) in enumerate(zip(sv_ord,lefts,widths)):
                ax.text(left+w/2,k,f'{sv:+.4f}',ha='center',va='center',
                        fontsize=8,color='white',fontweight='bold')
            plt.tight_layout()
            plt.savefig(os.path.join(base, f'SHAP_Waterfall_{label}_Sample{sidx:03d}.png'),
                        dpi=CONFIG['dpi'], bbox_inches='tight'); plt.close()
            print(f"      ✓ {label} 样本#{sidx}")
            cumulative_kq = []; c = base_value
            for sv in sv_ord:
                c+=sv; cumulative_kq.append(c)
            wf_sheets[f'Waterfall_{label}'] = pd.DataFrame({
                'Importance_Rank': range(1, len(sv_ord)+1), 'Feature': fn_ord,
                'Feature_Value': fv_ord, 'SHAP_Value': sv_ord, 'Abs_SHAP': np.abs(sv_ord),
                'Direction': ['Pos' if s>=0 else 'Neg' for s in sv_ord],
                'Bar_Left': lefts, 'Bar_Width': widths, 'Cumulative_KQ': cumulative_kq,
                'Baseline': base_value, 'Predicted_KQ': pred_val,
                'Sample_Index': sidx, 'Category': label,
            })
        except Exception as e:
            print(f"      ❌ {label} 失败: {e}"); plt.close('all')

    wf_sheets['Sample_Summary'] = pd.DataFrame({
        'Category': list(sample_map.keys()), 'Sample_Index': list(sample_map.values()),
        'Predicted_KQ': [pred_vals[s] for s in sample_map.values()], 'Baseline': base_value,
        'Sum_SHAP': [np.sum(shap_values[s]) for s in sample_map.values()],
        'Top1_Feature': [features[np.argmax(np.abs(shap_values[s]))] for s in sample_map.values()],
        'Top1_SHAP_Val': [shap_values[s][np.argmax(np.abs(shap_values[s]))] for s in sample_map.values()],
    })
    write_excel_multisheets(
        os.path.join(base, 'SHAP_Waterfall_Data.xlsx'), wf_sheets,
        {'baseline': float(base_value), 'n_samples': len(pred_vals), 'Format': 'Wide Table'})
    print(f"    ✓ SHAP_Waterfall_Data.xlsx（{len(wf_sheets)} Sheets）→ shap_waterfall/")

# ==============================================================
# 主分析函数
# ==============================================================
def analyze_gp_shap():
    print("\n"+"="*80)
    print("TOP1 GP模型 SHAP完整分析程序 v9")
    print("★ v9: 增强版Dependence图 → 精确Positive/Negative分区")
    print("      基于Lowess曲线与SHAP=0的实际交叉点划分区域")
    print("      交叉点处标注特征值数值（红线+红点+数值）")
    print("="*80)
    cleanup_matplotlib()

    out_dir = CONFIG['output_dir']
    create_output_dirs(out_dir)

    model_info = load_gp_model_and_data()
    features   = model_info['features']
    X_data, y_data = model_info['X_data'], model_info['y_data']
    print(f"\n模型: {model_info['model_name']}  样本数: {len(X_data)}")

    print(f"\n{'='*60}\n1️⃣  构建KernelExplainer\n{'='*60}")
    explainer = build_shap_explainer(model_info)

    print(f"\n{'='*60}\n2️⃣  计算SHAP值\n{'='*60}")
    shap_values, X_explain = compute_shap_values(explainer, model_info)

    print(f"\n{'='*60}\n3️⃣  Summary / Beeswarm / Bar / Violin\n{'='*60}")
    plot_shap_summary(shap_values, X_explain, features, out_dir)

    print(f"\n{'='*60}\n4️⃣  逐样本SHAP热图\n{'='*60}")
    plot_shap_sample_heatmap(shap_values, X_explain, features,
                              explainer, y_data, out_dir)

    print(f"\n{'='*60}\n5️⃣  单特征依赖图（原版）\n{'='*60}")
    plot_shap_dependence(shap_values, X_explain, features, out_dir)

    print(f"\n{'='*60}\n6️⃣  增强版Dependence图（★v9: 精确分区+交叉点标注）\n{'='*60}")
    plot_shap_dependence_enhanced(shap_values, X_explain, features,
                                   explainer, y_data, out_dir)

    print(f"\n{'='*60}\n7️⃣  特征交互热力图\n{'='*60}")
    plot_shap_interaction_heatmap(shap_values, X_explain, features,
                                   explainer, out_dir)

    print(f"\n{'='*60}\n8️⃣  Force Plot\n{'='*60}")
    plot_shap_force(shap_values, X_explain, features,
                    explainer, y_data, out_dir)

    print(f"\n{'='*60}\n9️⃣  Waterfall Plot\n{'='*60}")
    plot_shap_waterfall(shap_values, X_explain, features,
                         explainer, y_data, out_dir)

    print(f"\n{'='*80}")
    print("✅ SHAP v9 分析完成！")
    print(f"{'='*80}")
    print(f"\n  输出目录: {out_dir}")
    print(f"\n  ★ v9核心修改：")
    print(f"    精确分区：基于Lowess曲线与SHAP=0的实际交叉点")
    print(f"    正贡献区（浅红）/ 负贡献区（浅蓝）按曲线走向精确填充")
    print(f"    交叉点标注：红色虚线 + 红色圆点 + 特征值数值")
    print(f"    Excel新增：Precise_Regions Sheet + Zero_Crossings Sheet")

    cleanup_matplotlib()
    return True

if __name__ == "__main__":
    analyze_gp_shap()


TOP1 GP模型 SHAP完整分析程序 v9
★ v9: 增强版Dependence图 → 精确Positive/Negative分区
      基于Lowess曲线与SHAP=0的实际交叉点划分区域
      交叉点处标注特征值数值（红线+红点+数值）
✓ 输出目录已创建: D:\博二上\模型分析可视化\shap7
=== 加载TOP1高斯过程模型和数据 ===
✓ 模型: GP_Matern_0.5  特征: ['deltaP_热导率W/(mk)', 'x', 'Ec', 'Ω', 'deltaP_E13 Electron affinity', 'ΔSmix']
✓ 有效样本: 202

模型: GP_Matern_0.5  样本数: 202

1️⃣  构建KernelExplainer

  构建 SHAP KernelExplainer...
    ✓ 构建完成（背景50点）

2️⃣  计算SHAP值

  计算 SHAP 值...


  0%|          | 0/202 [00:00<?, ?it/s]

    ✓ 完成，shape: (202, 6)

3️⃣  Summary / Beeswarm / Bar / Violin

  生成 SHAP Summary Plot...
    ✓ 4张图已保存
    ✓ SHAP_Summary_Data.xlsx（5 Sheets）→ shap_summary/

4️⃣  逐样本SHAP热图

  生成 逐样本SHAP热图...
    ✓ 3张热图已保存
    ✓ SHAP_Heatmap_Data.xlsx（3 Sheets）→ shap_heatmap/

5️⃣  单特征依赖图（原版）

  生成 SHAP 单特征依赖图（原版）...
    处理: deltaP_热导率W/(mk)
      ✓ deltaP_热导率W/(mk) → shap_dependence/
    处理: x
      ✓ x → shap_dependence/
    处理: Ec
      ✓ Ec → shap_dependence/
    处理: Ω
      ✓ Ω → shap_dependence/
    处理: deltaP_E13 Electron affinity
      ✓ deltaP_E13 Electron affinity → shap_dependence/
    处理: ΔSmix
      ✓ ΔSmix → shap_dependence/

6️⃣  增强版Dependence图（★v9: 精确分区+交叉点标注）

  生成 增强版SHAP Dependence图（v9: 精确Positive/Negative分区 + 交叉点标注）...
    ✓ 合并大图已保存
      ✓ deltaP_热导率W/(mk)（1个交叉点）→ shap_dependence_enhanced/
      ✓ Ω（1个交叉点）→ shap_dependence_enhanced/
      ✓ x（1个交叉点）→ shap_dependence_enhanced/
      ✓ ΔSmix（4个交叉点）→ shap_dependence_enhanced/
      ✓ deltaP_E13 Electron affinity（2个交叉点）→ shap_depende